In [1]:
pip install selenium pillow pytesseract requests


In [12]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pytesseract
from PIL import Image
import io
import requests
import base64

driver = webdriver.Chrome()
driver.get("https://www.tender247.com/tender-details/89388410")

wait = WebDriverWait(driver, 15)

# (Optional) Click to expand "NIT Documents" if it's in a collapsed section
try:
    button = wait.until(
        EC.element_to_be_clickable((By.XPATH, "//h2[contains(text(), 'NIT Documents')]"))
    )
    button.click()
except Exception as e:
    print("Expand button not found or already open:", e)

# Wait for the image to appear (try different classnames if needed)
try:
    # Try alt text first
    img = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'img[alt="NIT"]')))
except:
    try:
        # If alt is not reliable, try by class name (update class if needed)
        img = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'img.sm.w-full')))
    except Exception as e:
        print("Image not found by any selector:", e)
        driver.quit()
        raise

# Remove blur using JavaScript
driver.execute_script("""
    var img = arguments[0];
    img.className = 'sm w-full';  // Remove blur-sm, keep normal class
    img.style.filter = 'none';    // Remove blur CSS directly if present
""", img)

# Wait a bit for the blur to be removed visually
import time
time.sleep(1)

# Grab image src
img_src = img.get_attribute('src')

# Download or decode the image
if img_src.startswith('data:image'):
    img_data = img_src.split(',')[1]
    image = Image.open(io.BytesIO(base64.b64decode(img_data)))
else:
    response = requests.get(img_src)
    image = Image.open(io.BytesIO(response.content))

# OCR
text = pytesseract.image_to_string(image)

# Extract Organisation Chain
import re
org_chain = None
for line in text.split('\n'):
    if "Organisation Chain" in line:
        org_chain = line
        break
print("Extracted Organisation Chain:", org_chain)
driver.quit()


Extracted Organisation Chain: Organisation Chain National Highways Authority of Indial|Head Office - NHAI|Technical - NHAI


In [9]:
import pytesseract
pytesseract.pytesseract.tesseract_cmd = r"C:\Users\Jaydeb\AppData\Local\Programs\Tesseract-OCR\tesseract.exe"


In [10]:
import os
import pytesseract

tess_path = r"C:\Users\Jaydeb\AppData\Local\Programs\Tesseract-OCR\tesseract.exe"
if os.path.isfile(tess_path):
    print("✅ Tesseract FOUND at", tess_path)
    pytesseract.pytesseract.tesseract_cmd = tess_path
    # Test OCR on a sample image
    # from PIL import Image
    # img = Image.open('your_image.png')
    # print(pytesseract.image_to_string(img))
else:
    print("❌ Not found at:", tess_path)


✅ Tesseract FOUND at C:\Users\Jaydeb\AppData\Local\Programs\Tesseract-OCR\tesseract.exe


In [1]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pytesseract
from PIL import Image
import io
import requests
import base64
import time
import random
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# SET your Tesseract path here
pytesseract.pytesseract.tesseract_cmd = r"C:\Users\Jaydeb\AppData\Local\Programs\Tesseract-OCR\tesseract.exe"

# Load your Excel sheet
input_excel = "Tender247_gujarat.xlsx"
output_excel = "Tender247_gujarat_with_organisation_chain.xlsx"

df = pd.read_excel(input_excel)
df['Organisation Chain'] = ""  # Add new column

# Setup Selenium
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 15)

# Setup Requests session with retries
session = requests.Session()
retries = Retry(total=3, backoff_factor=1, status_forcelist=[502, 503, 504])
session.mount('http://', HTTPAdapter(max_retries=retries))
session.mount('https://', HTTPAdapter(max_retries=retries))

# Only process the first 20 rows
for idx, row in df.head(20).iterrows():
    print(f"\nProcessing row {idx+1}/20 T247 ID: {row['T247 ID']}")

    # If you have a 'Bid Link' column, use it directly, else build the link:
    if 'Bid Link' in df.columns:
        link = row['Bid Link']
    else:
        link = f"https://www.tender247.com/tender-details/{row['T247 ID']}"
    
    try:
        driver.get(link)
        time.sleep(2)  # let page load

        # Try to expand "NIT Documents" section (if needed)
        try:
            button = wait.until(
                EC.element_to_be_clickable((By.XPATH, "//h2[contains(text(), 'NIT Documents')]"))
            )
            button.click()
            time.sleep(1)
        except Exception as e:
            print("Expand button not found or already open:", e)

        # Try to get the image
        try:
            try:
                img = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'img[alt="NIT"]')))
            except:
                img = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'img.sm.w-full')))
        except Exception as e:
            print("Image not found by any selector:", e)
            df.at[idx, 'Organisation Chain'] = "Not found"
            continue

        # Remove blur with JS
        driver.execute_script("""
            var img = arguments[0];
            img.className = 'sm w-full';
            img.style.filter = 'none';
        """, img)
        time.sleep(1)

        # Get image src and OCR
        img_src = img.get_attribute('src')
        if (img_src.startswith('http://localhost') or 
            img_src.startswith('http://127.0.0.1') or 
            img_src.startswith('https://localhost') or 
            img_src.startswith('https://127.0.0.1')):
            print("Invalid image src:", img_src)
            df.at[idx, 'Organisation Chain'] = "Invalid image src"
            continue

        try:
            if img_src.startswith('data:image'):
                img_data = img_src.split(',')[1]
                image = Image.open(io.BytesIO(base64.b64decode(img_data)))
            else:
                response = session.get(img_src, timeout=30)
                image = Image.open(io.BytesIO(response.content))
            
            text = pytesseract.image_to_string(image)

            # Extract Organisation Chain
            org_chain = ""
            for line in text.split('\n'):
                if "Organisation Chain" in line:
                    org_chain = line
                    break
            if not org_chain:
                org_chain = "Not found"
            print("Extracted:", org_chain)
            df.at[idx, 'Organisation Chain'] = org_chain
        except Exception as e:
            print("OCR/Image download error:", e)
            df.at[idx, 'Organisation Chain'] = f"OCR/Image error: {e}"
            continue

    except Exception as e:
        print("General error for this row:", e)
        df.at[idx, 'Organisation Chain'] = f"General error: {e}"
        continue

    # Be polite to the server
    time.sleep(random.uniform(2, 4))

# Done! Save the file (all rows, with only first 20 filled)
driver.quit()
df.to_excel(output_excel, index=False)
print("\n\nAll done! Results saved to:", output_excel)



Processing row 1/20 T247 ID: 89388410
Extracted: Organisation Chain National Highways Authority of Indial|Head Office - NHAI|Technical - NHAI

Processing row 2/20 T247 ID: 89879025
Extracted: Organisation Chain National Highways Authority of Indial|Head Office - NHAI|Technical - NHAI

Processing row 3/20 T247 ID: 89836145
Extracted: Organisation Chain National Highways Authority of Indial|Head Office - NHAI|Technical - NHAI

Processing row 4/20 T247 ID: 89879584
Extracted: Organisation Chain National Highways Authority of Indial|Head Office - NHAI|Technical - NHAI

Processing row 5/20 T247 ID: 89879317
Extracted: Organisation Chain National Highways Authority of Indial|Head Office - NHAI|Technical - NHAI

Processing row 6/20 T247 ID: 89879075
Extracted: Organisation Chain National Highways Authority of Indial|Head Office - NHAI|Technical - NHAI

Processing row 7/20 T247 ID: 89879453
Extracted: Organisation Chain National Highways Authority of Indial|Head Office - NHAI|Technical - NHAI

In [ ]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pytesseract
from PIL import Image
import io
import requests
import base64
import time
import random
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# SET your Tesseract path here
pytesseract.pytesseract.tesseract_cmd = r"C:\Users\Jaydeb\AppData\Local\Programs\Tesseract-OCR\tesseract.exe"

# Load your Excel sheet
input_excel = "Tender247_gujarat.xlsx"
output_excel = "Tender247_gujarat_with_organisation_chain.xlsx"

df = pd.read_excel(input_excel)
df['Organisation Chain'] = ""  # Add new column

# Setup Selenium
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 15)

# Setup Requests session with retries
session = requests.Session()
retries = Retry(total=3, backoff_factor=1, status_forcelist=[502, 503, 504])
session.mount('http://', HTTPAdapter(max_retries=retries))
session.mount('https://', HTTPAdapter(max_retries=retries))

for idx, row in df.iterrows():
    print(f"\nProcessing row {idx+1}/{len(df)} T247 ID: {row['T247 ID']}")

    # If you have a 'Bid Link' column, use it directly, else build the link:
    if 'Bid Link' in df.columns:
        link = row['Bid Link']
    else:
        link = f"https://www.tender247.com/tender-details/{row['T247 ID']}"
    
    try:
        driver.get(link)
        time.sleep(2)  # let page load

        # Try to expand "NIT Documents" section (if needed)
        try:
            button = wait.until(
                EC.element_to_be_clickable((By.XPATH, "//h2[contains(text(), 'NIT Documents')]"))
            )
            button.click()
            time.sleep(1)
        except Exception as e:
            print("Expand button not found or already open:", e)

        # Try to get the image
        try:
            try:
                img = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'img[alt="NIT"]')))
            except:
                img = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'img.sm.w-full')))
        except Exception as e:
            print("Image not found by any selector:", e)
            df.at[idx, 'Organisation Chain'] = "Not found"
            continue

        # Remove blur with JS
        driver.execute_script("""
            var img = arguments[0];
            img.className = 'sm w-full';
            img.style.filter = 'none';
        """, img)
        time.sleep(1)

        # Get image src and OCR
        img_src = img.get_attribute('src')
        if (img_src.startswith('http://localhost') or 
            img_src.startswith('http://127.0.0.1') or 
            img_src.startswith('https://localhost') or 
            img_src.startswith('https://127.0.0.1')):
            print("Invalid image src:", img_src)
            df.at[idx, 'Organisation Chain'] = "Invalid image src"
            continue

        try:
            if img_src.startswith('data:image'):
                img_data = img_src.split(',')[1]
                image = Image.open(io.BytesIO(base64.b64decode(img_data)))
            else:
                response = session.get(img_src, timeout=30)
                image = Image.open(io.BytesIO(response.content))
            
            text = pytesseract.image_to_string(image)

            # Extract Organisation Chain
            org_chain = ""
            for line in text.split('\n'):
                if "Organisation Chain" in line:
                    org_chain = line
                    break
            if not org_chain:
                org_chain = "Not found"
            print("Extracted:", org_chain)
            df.at[idx, 'Organisation Chain'] = org_chain
        except Exception as e:
            print("OCR/Image download error:", e)
            df.at[idx, 'Organisation Chain'] = f"OCR/Image error: {e}"
            continue

    except Exception as e:
        print("General error for this row:", e)
        df.at[idx, 'Organisation Chain'] = f"General error: {e}"
        continue

    # Be polite to the server
    time.sleep(random.uniform(2, 4))

# Done! Save the file.
driver.quit()
df.to_excel(output_excel, index=False)
print("\n\nAll done! Results saved to:", output_excel)



Processing row 1/640 T247 ID: 89388410


In [2]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pytesseract
from PIL import Image
import io
import requests
import base64
import time
import random
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# SET your Tesseract path here
pytesseract.pytesseract.tesseract_cmd = r"C:\Users\Jaydeb\AppData\Local\Programs\Tesseract-OCR\tesseract.exe"

# Load your Excel sheet
input_excel = "Tender247_gujarat.xlsx"
output_excel = "Tender247_gujarat_with_organisation_chain.xlsx"

df = pd.read_excel(input_excel)
df['Organisation Chain'] = ""             # New column
df['Tender Reference Number'] = ""        # New column
df['Tender ID'] = ""                      # New column

# Setup Selenium
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 15)

# Setup Requests session with retries
session = requests.Session()
retries = Retry(total=3, backoff_factor=1, status_forcelist=[502, 503, 504])
session.mount('http://', HTTPAdapter(max_retries=retries))
session.mount('https://', HTTPAdapter(max_retries=retries))

for idx, row in df.head(20).iterrows():
    print(f"\nProcessing row {idx+1}/20 T247 ID: {row['T247 ID']}")

    if 'Bid Link' in df.columns:
        link = row['Bid Link']
    else:
        link = f"https://www.tender247.com/tender-details/{row['T247 ID']}"

    try:
        driver.get(link)
        time.sleep(2)  # let page load

        # Try to expand "NIT Documents" section (if needed)
        try:
            button = wait.until(
                EC.element_to_be_clickable((By.XPATH, "//h2[contains(text(), 'NIT Documents')]"))
            )
            button.click()
            time.sleep(1)
        except Exception as e:
            print("Expand button not found or already open:", e)

        # Try to get the image
        try:
            try:
                img = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'img[alt=\"NIT\"]')))
            except:
                img = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'img.sm.w-full')))
        except Exception as e:
            print("Image not found by any selector:", e)
            df.at[idx, 'Organisation Chain'] = "Not found"
            continue

        # Remove blur with JS
        driver.execute_script("""
            var img = arguments[0];
            img.className = 'sm w-full';
            img.style.filter = 'none';
        """, img)
        time.sleep(1)

        # Get image src and OCR
        img_src = img.get_attribute('src')
        if (img_src.startswith('http://localhost') or 
            img_src.startswith('http://127.0.0.1') or 
            img_src.startswith('https://localhost') or 
            img_src.startswith('https://127.0.0.1')):
            print("Invalid image src:", img_src)
            df.at[idx, 'Organisation Chain'] = "Invalid image src"
            continue

        try:
            if img_src.startswith('data:image'):
                img_data = img_src.split(',')[1]
                image = Image.open(io.BytesIO(base64.b64decode(img_data)))
            else:
                response = session.get(img_src, timeout=30)
                image = Image.open(io.BytesIO(response.content))
            
            text = pytesseract.image_to_string(image)

            # Extract Organisation Chain, Tender Reference Number, Tender ID
            org_chain = ""
            tender_ref_no = ""
            tender_id = ""

            for line in text.split('\n'):
                if "Organisation Chain" in line:
                    org_chain = line
                if "Tender Reference Number" in line:
                    tender_ref_no = line
                if "Tender ID" in line:
                    tender_id = line

            if not org_chain:
                org_chain = "Not found"
            if not tender_ref_no:
                tender_ref_no = "Not found"
            if not tender_id:
                tender_id = "Not found"

            print("Extracted:", org_chain, tender_ref_no, tender_id)
            df.at[idx, 'Organisation Chain'] = org_chain
            df.at[idx, 'Tender Reference Number'] = tender_ref_no
            df.at[idx, 'Tender ID'] = tender_id

        except Exception as e:
            print("OCR/Image download error:", e)
            df.at[idx, 'Organisation Chain'] = f"OCR/Image error: {e}"
            df.at[idx, 'Tender Reference Number'] = f"OCR/Image error: {e}"
            df.at[idx, 'Tender ID'] = f"OCR/Image error: {e}"
            continue

    except Exception as e:
        print("General error for this row:", e)
        df.at[idx, 'Organisation Chain'] = f"General error: {e}"
        df.at[idx, 'Tender Reference Number'] = f"General error: {e}"
        df.at[idx, 'Tender ID'] = f"General error: {e}"
        continue

    # Be polite to the server
    time.sleep(random.uniform(2, 4))

# Done! Save the file (all rows, with only first 20 filled)
driver.quit()
df.to_excel(output_excel, index=False)
print("\n\nAll done! Results saved to:", output_excel)



Processing row 1/20 T247 ID: 89388410
Extracted: Organisation Chain National Highways Authority of Indial|Head Office - NHAI|Technical - NHAI Tender Reference Number NHAI/Tech/MH/S-N-A-S/ Pkg-V1/2024 Tender ID 2024 NHAI 2068921 Withdrawal Allowed ‘Yes

Processing row 2/20 T247 ID: 89879025
Extracted: Organisation Chain National Highways Authority of Indial|Head Office - NHAI|Technical - NHAI Tender Reference Number NHAI/Tech/MH/S-N-A-S/ Pkg-V1/2024 Tender ID 2024 NHAI 2068921 Withdrawal Allowed ‘Yes

Processing row 3/20 T247 ID: 89836145
Extracted: Organisation Chain National Highways Authority of Indial|Head Office - NHAI|Technical - NHAI Tender Reference Number NHAI/Tech/MH/S-N-A-S/ Pkg-V1/2024 Tender ID 2024 NHAI 2068921 Withdrawal Allowed ‘Yes

Processing row 4/20 T247 ID: 89879584
Extracted: Organisation Chain National Highways Authority of Indial|Head Office - NHAI|Technical - NHAI Tender Reference Number NHAI/Tech/MH/S-N-A-S/ Pkg-V1/2024 Tender ID 2024 NHAI 2068921 Withdrawal A